In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# STEP 1: LOAD BRONZE TABLES

air = spark.table("smart_city.pollution.bronze_air_quality")
weather = spark.table("smart_city.pollution.bronze_weather")
nasa = spark.table("smart_city.pollution.bronze_nasa_power")

print("All Bronze tables loaded!")

All Bronze tables loaded!


In [0]:
# STEP 2: DEDUPLICATION FUNCTION

def keep_latest_record(df, keys):

    if "extracted_at_utc" in df.columns:
        order_col = F.col("extracted_at_utc").desc_nulls_last()
    else:
        order_col = F.col("ingestion_timestamp").desc_nulls_last()

    w = Window.partitionBy(*keys).orderBy(order_col)

    return (
        df.withColumn("_rn", F.row_number().over(w))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )


In [0]:

# STEP 3: CLEAN AIR QUALITY

air_silver = air.filter(
    F.col("city").isNotNull() &
    F.col("timestamp").isNotNull()
)

air_silver = air_silver.withColumn(
    "city", F.initcap(F.trim("city"))
)

air_silver = air_silver.withColumn(
    "timestamp", F.to_timestamp("timestamp")
)

air_silver = air_silver.filter(
    F.col("timestamp").isNotNull()
)

air_columns = [
    "pm10", "pm2_5", "nitrogen_dioxide",
    "sulphur_dioxide", "carbon_monoxide", "ozone"
]

for c in air_columns:
    air_silver = air_silver.withColumn(
        c, F.col(c).cast("double")
    )
    air_silver = air_silver.withColumn(
        c,
        F.when(F.col(c) >= 0, F.col(c)).otherwise(None)
    )

air_silver = keep_latest_record(
    air_silver, ["city", "timestamp"]
)

air_silver = air_silver.withColumn(
    "date", F.to_date("timestamp")
)


In [0]:

# STEP 4: CLEAN WEATHER

weather_silver = weather.filter(
    F.col("city").isNotNull() &
    F.col("timestamp").isNotNull()
)

weather_silver = weather_silver.withColumn(
    "city", F.initcap(F.trim("city"))
)

weather_silver = weather_silver.withColumn(
    "timestamp", F.to_timestamp("timestamp")
)

weather_silver = weather_silver.filter(
    F.col("timestamp").isNotNull()
)

for c in ["temperature", "humidity", "wind_speed", "precipitation"]:
    weather_silver = weather_silver.withColumn(
        c, F.col(c).cast("double")
    )

weather_silver = weather_silver.withColumn(
    "humidity",
    F.when(F.col("humidity").between(0, 100),
           F.col("humidity")).otherwise(None)
)

for c in ["wind_speed", "precipitation"]:
    weather_silver = weather_silver.withColumn(
        c,
        F.when(F.col(c) >= 0, F.col(c)).otherwise(None)
    )

weather_silver = keep_latest_record(
    weather_silver, ["city", "timestamp"]
)

weather_silver = weather_silver.withColumn(
    "date", F.to_date("timestamp")
)


In [0]:

# STEP 5: CLEAN NASA POWER

nasa_silver = nasa.filter(
    F.col("city").isNotNull() &
    F.col("date").isNotNull()
)

nasa_silver = nasa_silver.withColumn(
    "city", F.initcap(F.trim("city"))
)

nasa_silver = nasa_silver.withColumn(
    "date",
    F.to_date(F.col("date"))
)

for c in [
    "temperature_avg", "temperature_max",
    "temperature_min", "humidity",
    "wind_speed", "precipitation"
]:
    nasa_silver = nasa_silver.withColumn(
        c, F.col(c).cast("double")
    )

nasa_silver = keep_latest_record(
    nasa_silver, ["city", "date"]
)



In [0]:
# STEP 6: DATA QUALITY VALIDATION

datasets = {
    "Air Quality": air_silver,
    "Weather": weather_silver,
    "NASA POWER": nasa_silver
}

for name, df in datasets.items():

    count = df.count()

    print(name, "records:", count)

    if count == 0:
        raise Exception(name + " Silver table is empty!")



Air Quality records: 1008
Weather records: 1008
NASA POWER records: 42


In [0]:
# STEP 7: SAVE SILVER TABLES

air_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_city.pollution.silver_air_quality")

weather_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_city.pollution.silver_weather")

nasa_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_city.pollution.silver_nasa_power")


In [0]:

# STEP 8: FINAL VERIFICATION

print("SILVER LAYER COMPLETED SUCCESSFULLY!")

for table in [
    "silver_air_quality",
    "silver_weather",
    "silver_nasa_power"
]:
    df = spark.table("smart_city.pollution." + table)
    print(table, ":", df.count())
    df.printSchema()


SILVER LAYER COMPLETED SUCCESSFULLY!
silver_air_quality : 1008
root
 |-- city: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm2_5: double (nullable = true)
 |-- nitrogen_dioxide: double (nullable = true)
 |-- sulphur_dioxide: double (nullable = true)
 |-- carbon_monoxide: double (nullable = true)
 |-- ozone: double (nullable = true)
 |-- source: string (nullable = true)
 |-- date: date (nullable = true)
 |-- extracted_at_utc: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)

silver_weather : 1008
root
 |-- city: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- source: string (nullable = true)
 |-- date: date (nullable = true)
 |-- extracted_at_utc: times